# is-differentiable-flag — worked example 1: Non-differentiable op produces no Recipe and requires_grad=False

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `is-differentiable-flag`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

When `wrap_forward_fn` is called with `is_differentiable=False`, the wrapped op should never produce a tracked output regardless of its inputs. This means `requires_grad` on the output is always `False`, and no `Recipe` object is constructed. This pattern applies to ops like `argmax` or `equal` that have no meaningful gradient.

## Worked solution

Step 1: We define a minimal `MiniTensor` class that holds an `array` (a numpy array), a `requires_grad` flag, and a `recipe` slot.

Step 2: We implement `wrap_forward_fn(fwd_fn, is_differentiable=True)`. Inside `tensor_func`, we unbox any `MiniTensor` args to their raw arrays, call `fwd_fn`, and compute `requires_grad` as the three-gate AND: `grad_tracking_enabled AND is_differentiable AND any-tracked-input`.

Step 3: When `is_differentiable=False`, the second gate short-circuits the AND to `False` regardless of the other gates. So `requires_grad=False` always, and we skip the `Recipe` construction — `out.recipe` stays `None`.

Step 4: We wrap a fake `np.equal`-like function with `is_differentiable=False` and call it on a tracked input. We verify `out.requires_grad is False` and `out.recipe is None`.

In [ ]:
import numpy as np
from dataclasses import dataclass
from typing import Any, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Any
    args: tuple
    kwargs: dict
    parents: dict

class MiniTensor:
    def __init__(self, array, requires_grad=False):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe: Optional[Recipe] = None

def wrap_forward_fn(fwd_fn, is_differentiable=True):
    def tensor_func(*args, **kwargs):
        raw_args = tuple(a.array if isinstance(a, MiniTensor) else a for a in args)
        out_arr = fwd_fn(*raw_args, **kwargs)
        requires_grad = (
            grad_tracking_enabled
            and is_differentiable
            and any(isinstance(a, MiniTensor) and a.requires_grad for a in args)
        )
        out = MiniTensor(out_arr, requires_grad)
        if requires_grad:
            parents = {i: a for i, a in enumerate(args) if isinstance(a, MiniTensor)}
            out.recipe = Recipe(fwd_fn, raw_args, kwargs, parents)
        return out
    return tensor_func

# Demonstrate: wrap a non-differentiable op
def np_equal(a, b):
    return (a == b).astype(float)

wrapped_eq = wrap_forward_fn(np_equal, is_differentiable=False)

x = MiniTensor(np.array([1.0, 2.0, 3.0]), requires_grad=True)
y = MiniTensor(np.array([1.0, 0.0, 3.0]), requires_grad=False)

out = wrapped_eq(x, y)
print('requires_grad:', out.requires_grad)  # False
print('recipe:', out.recipe)               # None
print('output values:', out.array)         # [1. 0. 1.]

# Even if BOTH inputs are tracked, is_differentiable=False overrides
y2 = MiniTensor(np.array([1.0, 0.0, 3.0]), requires_grad=True)
out2 = wrapped_eq(x, y2)
print('both-tracked requires_grad:', out2.requires_grad)  # False
print('both-tracked recipe:', out2.recipe)                # None